# PaliGemma2 Fine-tuning — Lokálny dataset (person/weapon detection)

Notebook upravený pre server s **32-core CPU, 128GB RAM, NVIDIA L40S 48GB**.

Dataset formát: JSONL s lokálnymi obrázkami.

## 1. Inštalácia závislostí

In [1]:
!pip install -q -U datasets bitsandbytes peft transformers accelerate Pillow

## 2. Konfigurácia — uprav tu svoje cesty

In [2]:
# ==================== KONFIGURÁCIA ====================
DATASET_PATH  = "PERSON-WEAPON-FINISHED-1/dataset/_annotations.train.jsonl"
IMAGE_FOLDER  = "PERSON-WEAPON-FINISHED-1/dataset/"
MODEL_ID      = "google/paligemma2-3b-pt-224"

OUTPUT_DIR    = "paligemma2_weapon_det_suffix_fixed"
HF_REPO_ID    = None   # napr. "tvoj-username/paligemma2-weapon" alebo None ak nechceš pushnúť

# Tréningové hyperparametre
NUM_EPOCHS    = 5
BATCH_SIZE    = 4      # L40S 48GB zvládne bez problémov
GRAD_ACCUM    = 8      # efektívny batch = BATCH_SIZE * GRAD_ACCUM = 8
LEARNING_RATE = 2e-5
USE_LORA      = True   # True = LoRA (odporúčané), False = full fine-tune text dekodera
# =======================================================

## 3. Načítanie a príprava datasetu

In [3]:
import json
import os
import re
from pathlib import Path
from PIL import Image

def load_jsonl_dataset(jsonl_path: str, image_folder: str):
    """
    Načíta JSONL dataset a dynamicky rozdelí prompty na person, weapon a kombinované.
    """
    records = []
    image_folder = Path(image_folder)
    
    # Regex pre nájdenie PaliGemma bounding boxov a ich tried.
    # Zachytí presne 4x <loc...> a následne názov triedy (napr. "person" alebo "weapon")
    pattern = r'((?:<loc\d{4}>){4}\s*([a-zA-Z0-9_-]+))'
    
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            
            try:
                record = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"  [WARN] Riadok {line_num} preskočený (JSON chyba): {e}")
                continue
            
            # Získanie ciest a pôvodného suffixu
            image_file = record.get("image") or record.get("file_name") or record.get("filename")
            original_suffix = record.get("suffix") or record.get("answer") or record.get("label") or ""
            
            if not image_file:
                continue
            
            img_path = image_folder / image_file
            if not img_path.exists():
                print(f"  [WARN] Riadok {line_num}: obrázok nenájdený: {img_path}")
                continue
            
            # --- NOVÁ LOGIKA PRE ROZDELENIE PROMPTOV ---
            # 1. Vyextrahujeme všetky anotácie z pôvodného suffixu
            matches = re.findall(pattern, original_suffix)
            
            # matches obsahuje tuples v tvare ('<loc..><loc..><loc..><loc..> person', 'person')
            person_annotations = "".join([m[0] for m in matches if m[1].lower() == 'person'])
            weapon_annotations = "".join([m[0] for m in matches if m[1].lower() == 'weapon'])
            both_annotations = person_annotations + weapon_annotations
            
            # 2. Vytvoríme 3 samostatné záznamy pre tento obrázok
            
            # A) Prompt iba pre osobu
            records.append({
                "image_path": str(img_path),
                "prefix": "detect person",
                "suffix": person_annotations
            })
            
            # B) Prompt iba pre zbraň
            records.append({
                "image_path": str(img_path),
                "prefix": "detect weapon",
                "suffix": weapon_annotations
            })
            
            # C) Kombinovaný prompt (pre zachovanie pôvodného správania ako bonus)
            records.append({
                "image_path": str(img_path),
                "prefix": "detect person ; weapon",
                "suffix": both_annotations
            })
            # ---------------------------------------------
    
    print(f"Načítaných trénovacích záznamov po augmentácii promptov: {len(records)}")
    return records


all_records = load_jsonl_dataset(DATASET_PATH, IMAGE_FOLDER)

# Ukážka prvého záznamu
print("\nUkážka záznamu:")
print(json.dumps({k: v for k, v in all_records[0].items() if k != 'image_path'}, ensure_ascii=False, indent=2))
print("image_path:", all_records[0]['image_path'])

Načítaných trénovacích záznamov po augmentácii promptov: 71475

Ukážka záznamu:
{
  "prefix": "detect person",
  "suffix": "<loc0434><loc0338><loc0512><loc0412> person<loc0574><loc0498><loc0834><loc0606> person"
}
image_path: PERSON-WEAPON-FINISHED-1/dataset/aug_mixed_005223_jpg.rf.a93305fbd00dab3d5de87069bffdad33.jpg


In [4]:
import random

random.seed(42)
random.shuffle(all_records)

split_idx  = int(len(all_records) * 0.9)
train_data = all_records[:split_idx]
val_data   = all_records[split_idx:]

print(f"Train: {len(train_data)} | Val: {len(val_data)}")

Train: 64327 | Val: 7148


## 4. Načítanie modelu a processora

In [5]:
import torch
from transformers import PaliGemmaProcessor, PaliGemmaForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Použité zariadenie: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

processor = PaliGemmaProcessor.from_pretrained(MODEL_ID)
print("Processor načítaný.")

Processor načítaný.


In [7]:
if USE_LORA:
    # ── LoRA fine-tuning (odporúčané — rýchle, menej VRAM) ──
    from peft import get_peft_model, LoraConfig

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "o_proj", "k_proj", "v_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        task_type="CAUSAL_LM",
    )

    model = PaliGemmaForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

else:
    # ── Full fine-tune iba text dekodera (zmrazí vision + projector) ──
    model = PaliGemmaForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
    ).to(device)

    for param in model.vision_tower.parameters():
        param.requires_grad = False
    for param in model.multi_modal_projector.parameters():
        param.requires_grad = False

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Trénovateľné parametre: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

DTYPE = model.dtype
print(f"Model dtype: {DTYPE}")

trainable params: 23,752,704 || all params: 3,055,995,120 || trainable%: 0.7772
Model dtype: torch.bfloat16


## 5. Collate funkcia

In [8]:
from PIL import Image

image_token = processor.tokenizer.convert_tokens_to_ids("<image>")

def collate_fn(examples):
    """
    Každý example = dict s kľúčmi: image_path, prefix, suffix
    
    PaliGemma prompt formát:
      '<image>' + prefix  →  suffix
    """
    images = []
    texts  = []
    labels = []

    for ex in examples:
        try:
            img = Image.open(ex["image_path"]).convert("RGB")
        except Exception as e:
            print(f"  [WARN] Chyba pri načítaní obrázka {ex['image_path']}: {e}")
            continue

        images.append(img)
        texts.append("<image>" + ex["prefix"])
        labels.append(ex["suffix"])

    if not images:
        return None

    tokens = processor(
        text=texts,
        images=images,
        suffix=labels,
        return_tensors="pt",
        padding="longest",
    )

    # POZOR: .to(device) tu NESMIE byť!
    # DataLoader worker procesy sú forknuté a nemôžu inicializovať CUDA.
    # Trainer automaticky presunie batch na GPU pred forward passom.
    # Konvertujeme iba dtype (bfloat16) — to je CPU operácia, bezpečná vo workeroch.
    return tokens.to(DTYPE)

print("Collate funkcia pripravená.")

# Rýchly test
test_batch = collate_fn(train_data[:2])
if test_batch:
    print("Test batch OK — input_ids shape:", test_batch["input_ids"].shape)
    print("Test batch dtype:", test_batch["input_ids"].dtype, "| device:", test_batch["input_ids"].device)
else:
    print("[ERROR] Test batch zlyhal — skontroluj cesty k obrázkom!")

Collate funkcia pripravená.
Test batch OK — input_ids shape: torch.Size([2, 268])
Test batch dtype: torch.int64 | device: cpu


## 6. Training argumenty

In [9]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    weight_decay=1e-6,
    adam_beta2=0.999,
    warmup_ratio=0.03,
    bf16=True,                       # L40S podporuje bf16
    tf32=True,                       # dodatočné zrýchlenie na Ampere/Ada GPU
    optim="adamw_torch_fused",       # fused optimizer — rýchlejší na GPU
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    remove_unused_columns=False,     # DÔLEŽITÉ pre custom collate_fn
    dataloader_num_workers=0,        # MUSÍ byť 0! Workers sú forknuté procesy
                                     # a nemôžu reinicializovať CUDA (RuntimeError)
    dataloader_pin_memory=False,     # pin_memory tiež vyžaduje CUDA vo workeroch → vypnúť
    report_to=["tensorboard"],
    run_name="paligemma2-weapon-detection",
)

print("TrainingArguments pripravené.")
print(f"  Efektívny batch size: {BATCH_SIZE * GRAD_ACCUM} (per-device: {BATCH_SIZE}, grad_accum: {GRAD_ACCUM})")

TrainingArguments pripravené.
  Efektívny batch size: 32 (per-device: 4, grad_accum: 8)


## 7. Tréning

In [10]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=collate_fn,
)

print("Trainer inicializovaný. Spúšťam tréning...")
trainer.train()

TrainOutput(global_step=10055, training_loss=2.5782994149396337, metrics={'train_runtime': 23417.1965, 'train_samples_per_second': 13.735, 'train_steps_per_second': 0.429, 'total_flos': 1.3227722477133125e+18, 'train_loss': 2.5782994149396337, 'epoch': 5.0})

## 8. Uloženie modelu

In [11]:
# Uloží lokálne
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"Model uložený do: {OUTPUT_DIR}")

# Push na HuggingFace Hub (ak je HF_REPO_ID nastavené)
if HF_REPO_ID:
    from huggingface_hub import login
    import os
    from dotenv import load_dotenv
    load_dotenv()
    login(token=os.getenv("HF_TOKEN"))
    
    trainer.push_to_hub(HF_REPO_ID)
    processor.push_to_hub(HF_REPO_ID)
    print(f"Model pushnutý na Hub: {HF_REPO_ID}")
else:
    print("HF_REPO_ID nie je nastavené — model ostáva iba lokálne.")

Model uložený do: paligemma2_weapon_det_suffix_fixed
HF_REPO_ID nie je nastavené — model ostáva iba lokálne.


## 9. Inferencia — test natrénovaného modelu

In [2]:
import torch
from transformers import PaliGemmaProcessor, PaliGemmaForConditionalGeneration
from peft import PeftModel
from PIL import Image

# Načítaj natrénovaný model
inf_processor = PaliGemmaProcessor.from_pretrained(OUTPUT_DIR)

if USE_LORA:
    base_model = PaliGemmaForConditionalGeneration.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
    )
    inf_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
else:
    inf_model = PaliGemmaForConditionalGeneration.from_pretrained(
        OUTPUT_DIR, torch_dtype=torch.bfloat16, device_map="auto"
    )

inf_model.eval()

def predict(image_path: str, prompt: str = "detect person ; weapon") -> str:
    img = Image.open(image_path).convert("RGB")
    inputs = inf_processor(
        text="<image>" + prompt,
        images=img,
        return_tensors="pt"
    ).to(torch.bfloat16).to(device)

    with torch.no_grad():
        output_ids = inf_model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
        )

    # Dekóduj iba nové tokeny (nie prompt)
    generated = output_ids[0][inputs["input_ids"].shape[-1]:]
    return inf_processor.decode(generated, skip_special_tokens=True)


# Test na prvom validačnom obrázku
if val_data:
    sample = val_data[0]
    prediction = predict(sample["image_path"], sample["prefix"])
    print(f"Obrázok : {sample['image_path']}")
    print(f"Prompt  : {sample['prefix']}")
    print(f"Očakáva : {sample['suffix']}")
    print(f"Predikcia: {prediction}")

NameError: name 'OUTPUT_DIR' is not defined

---
## Poznámky k JSONL formátu

Notebook predpokladá tento formát každého riadku v `.jsonl`:

```json
{"image": "img_001.jpg", "prefix": "detect person ; weapon", "suffix": "<loc0123><loc0456>person"}
```

**Ak máš iný formát**, uprav sekciu `# Mapovanie polí` v bunke 3 (`load_jsonl_dataset`). Bežné alternatívy:

| Tvoj kľúč | Mapuje na |
|---|---|
| `file_name` / `filename` | `image` |
| `question` / `prompt` | `prefix` |
| `answer` / `label` | `suffix` |

In [1]:

prediction = predict(sample["/home/xpekarcik/video-based-action-recognition/djangoweb/src/train/w001_converted_frame_00001.jpg"], prompt="detect weapon")

NameError: name 'predict' is not defined

In [3]:
import torch
from transformers import PaliGemmaProcessor, PaliGemmaForConditionalGeneration
from peft import PeftModel
from PIL import Image
import os

# NASTAV TIETO CESTY PRESNE PODĽA SEBA
MODEL_ID   = "google/paligemma2-3b-pt-224"
OUTPUT_DIR = "paligemma2_weapon_det_suffix_fixed" # Názov zložky, kam si model ukladal
device     = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Pripravené na zariadení: {device}")

Pripravené na zariadení: cuda


In [4]:
# 1. Načítaj processor zo zložky s výsledkom
processor = PaliGemmaProcessor.from_pretrained(OUTPUT_DIR)

# 2. Načítaj základný model (v bfloat16 pre úsporu pamäte)
base_model = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# 3. Pripoj k nemu tvoje natrénované LoRA váhy z disku
model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
model.eval()

print("Model úspešne načítaný z disku!")

Model úspešne načítaný z disku!


In [19]:
def quick_test(img_path, prompt):
    img = Image.open(img_path).convert("RGB")
    inputs = processor(text="<image>" + prompt, images=img, return_tensors="pt").to(torch.bfloat16).to(device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=100, do_sample=False)

    # Dekódovanie výsledku
    generated_text = processor.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return generated_text

# SEM DAJ CESTU K TESTOVACIEMU OBRÁZKU
test_img = "/home/xpekarcik/video-based-action-recognition/djangoweb/src/train/w065_converted_frame_00002.jpg"

print("Test 1 (Iba zbrane):", quick_test(test_img, "detect weapon;"))
print("Test 2 (Iba osoby):", quick_test(test_img, "detect person;"))

Test 2 (Iba osoby): <loc0348><loc0310><loc0733><loc0476> person<loc0196><loc0216><loc0536><loc0308> person<loc0580><loc0214><loc0900><loc0382> person<loc0423><loc0570><loc0624><loc0708> person<loc0295><loc0633><loc0448><loc0724> person<loc0342><loc0720><loc0536><loc0792> person<loc0423><loc0683><loc0588><loc0760> person
